In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://enhantica.github.io/edi-download/pypi/simple/


# Joint fit: PbSO₄, neutron and X-ray

Fit PbSO₄ to two patterns at once, the Rietveld round robin's neutron pattern from D1A at ILL
and its laboratory X-ray pattern: one structure, two experiments.

The project is the catalogue case `pd-neut-xray-cwl_pbso4-d1a-lab`, the one the command line and
the app's Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Joint</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">xray</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-xray-cwl_pbso4-d1a-lab', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['pbso4']
d1a = project.experiments['d1a']
lab = project.experiments['lab']

Data 'pd-neut-xray-cwl_pbso4-d1a-lab' is at data/pd-neut-xray-cwl_pbso4-d1a-lab-ce92206a5a9d/pd-neut-xray-cwl_pbso4-d1a-lab


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('d1a')

View(html=<378,972 characters>, height=480)

## Perform the analysis

Both patterns fit together. diffraction-lib's tutorial of this case frees everything at once,
and also the D1A wavelength, which this case keeps fixed.

### Scales, 2θ zeros and cell

In [6]:
d1a.linked_structures['pbso4'].scale.free = True
lab.linked_structures['pbso4'].scale.free = True
d1a.instrument.calib_twotheta_offset.free = True
lab.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,pbso4,cell,,length_a,8.47,,0,30,Å
2,pbso4,cell,,length_b,5.39,,0,30,Å
3,pbso4,cell,,length_c,6.95,,0,30,Å
4,d1a,instrument,,calib_twotheta_offset,0,,-inf,inf,deg
5,d1a,linked_structure,pbso4,scale,1.5,,-inf,inf,
6,lab,instrument,,calib_twotheta_offset,0,,-inf,inf,deg
7,lab,linked_structure,pbso4,scale,0.001,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 7 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       64.60       241.50
      1      0.046     32.57        61.40   74.6% ↓
      2      0.086     29.36        49.90   18.7% ↓
      3      0.126     28.47        46.91   6.0% ↓
      4      0.166     27.86        44.92   4.2% ↓
      5      0.206     27.34        43.27   3.7% ↓
      6      0.246     26.85        41.72   3.6% ↓
      7      0.282     26.38        40.29   3.4% ↓
      8      0.318     25.94        38.96   3.3% ↓
      9      0.356     25.50        37.63   3.4% ↓
     10      0.393     25.02        36.22   3.7% ↓
     11      0.432     24.52        34.79   4.0% ↓
     12      0.472     24.01        33.36   4.1% ↓
     13      0.512     23.47        31.88   4.4% ↓
     14      0.552     22.90        30.35   4.8% ↓
     15      0.591     22.34        28.88   4.9% ↓
     16      0.627     21.80

View(html=<87,263 characters>, height=376)

In [9]:
project.display.pattern('d1a')

View(html=<379,169 characters>, height=480)

In [10]:
project.display.pattern('lab')

View(html=<735,506 characters>, height=480)

### Backgrounds

D1A's points and the laboratory pattern's Chebyshev terms.

In [11]:
for point in d1a.background:
    point.intensity.free = True
for term in lab.background_terms:
    term.coef.free = True

In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 21 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       20.03        23.25
      1      0.056     19.11        21.17   9.0% ↓
      2      0.093     19.11        21.17   
      3      0.130     19.11        21.17   
      4      0.185     19.11        21.17   
  done · converged · 4 iters · reduced χ² 21.17 · Rwp 19.11 % · 0.23 s


View(html=<100,861 characters>, height=376)

In [13]:
project.display.pattern('d1a')

View(html=<405,224 characters>, height=480)

In [14]:
project.display.pattern('lab')

View(html=<787,385 characters>, height=480)

### Peak widths

In [15]:
d1a.peak.broad_gauss_u.free = True
d1a.peak.broad_gauss_v.free = True
d1a.peak.broad_gauss_w.free = True
d1a.peak.mixing_eta_0.free = True
lab.peak.broad_gauss_u.free = True
lab.peak.broad_gauss_v.free = True
lab.peak.broad_gauss_w.free = True
lab.peak.mixing_eta_0.free = True

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 29 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       19.11        21.19
      1      0.060     13.96        11.31   46.6% ↓
      2      0.109     12.37         8.88   21.5% ↓
      3      0.170     11.88         8.18   7.8% ↓
      4      0.243     11.62         7.84   4.2% ↓
      5      0.323     11.49         7.66   2.3% ↓
      6      0.409     11.42         7.57   1.1% ↓
      7      0.497     11.40         7.54   0.4% ↓
      8      0.586     11.39         7.53   0.1% ↓
      9      0.675     11.39         7.53   
     10      0.766     11.39         7.53   
     11      0.856     11.39         7.53   
     12      0.990     11.39         7.53   
  done · converged · 12 iters · reduced χ² 7.53 · Rwp 11.39 % · 1.10 s


View(html=<110,024 characters>, height=376)

In [17]:
project.display.pattern('d1a')

View(html=<405,258 characters>, height=480)

In [18]:
project.display.pattern('lab')

View(html=<787,893 characters>, height=480)

### Peak asymmetry, D1A

The D1A pattern's four Bérar-Baldinozzi terms.

In [19]:
d1a.peak.asym_beba_a0.free = True
d1a.peak.asym_beba_b0.free = True
d1a.peak.asym_beba_a1.free = True
d1a.peak.asym_beba_b1.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 33 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.39         7.53
      1      0.135     10.50         6.40   15.1% ↓
      2      0.224      9.87         5.66   11.6% ↓
      3      0.301      9.70         5.46   3.4% ↓
      4      0.372      9.65         5.40   1.1% ↓
      5      0.437      9.64         5.39   0.2% ↓
      6      0.500      9.64         5.39   
      7      0.563      9.64         5.39   
      8      0.626      9.64         5.39   
      9      0.718      9.64         5.39   
  done · converged · 9 iters · reduced χ² 5.39 · Rwp 9.64 % · 0.80 s


View(html=<114,131 characters>, height=376)

In [21]:
project.display.pattern('d1a')

View(html=<405,329 characters>, height=480)

In [22]:
project.display.pattern('lab')

View(html=<787,989 characters>, height=480)

### Peak asymmetry, laboratory

The laboratory pattern's first-order terms.

In [23]:
lab.peak.asym_beba_a0.free = True
lab.peak.asym_beba_b0.free = True

In [24]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 35 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        9.64         5.39
      1      0.096      9.46         5.20   3.7% ↓
      2      0.161      9.45         5.19   0.2% ↓
      3      0.227      9.45         5.19   
      4      0.294      9.45         5.19   
      5      0.360      9.45         5.19   
      6      0.426      9.45         5.19   
      7      0.492      9.45         5.19   
      8      0.598      9.45         5.19   
  done · converged · 8 iters · reduced χ² 5.19 · Rwp 9.45 % · 0.67 s


View(html=<116,306 characters>, height=376)

In [25]:
project.display.pattern('d1a')

View(html=<405,278 characters>, height=480)

In [26]:
project.display.pattern('lab')

View(html=<788,218 characters>, height=480)

### Atom positions

The neutron and X-ray patterns together see the oxygen as well as the lead, so all free
coordinates fit at once.

In [27]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [28]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 46 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        9.45         5.19
      1      0.134      8.48         4.18   19.5% ↓
      2      0.228      8.47         4.17   0.2% ↓
      3      0.322      8.47         4.17   
      4      0.445      8.47         4.17   
  done · converged · 4 iters · reduced χ² 4.17 · Rwp 8.47 % · 0.55 s


View(html=<127,912 characters>, height=376)

In [29]:
project.display.pattern('d1a')

View(html=<405,463 characters>, height=480)

In [30]:
project.display.pattern('lab')

View(html=<788,131 characters>, height=480)

### Displacements

In [31]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [32]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 51 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.47         4.17
      1      0.113      6.05         2.13   49.0% ↓
      2      0.179      5.80         1.96   8.0% ↓
      3      0.245      5.80         1.96   0.1% ↓
      4      0.310      5.80         1.96   
      5      0.399      5.80         1.96   
  done · converged · 5 iters · reduced χ² 1.96 · Rwp 5.80 % · 0.46 s


View(html=<133,299 characters>, height=376)

In [33]:
project.display.pattern('d1a')

View(html=<405,681 characters>, height=480)

In [34]:
project.display.pattern('lab')

View(html=<788,238 characters>, height=480)

In [35]:
project.display.fit.correlations()

,1,2,3,4,5
1 d1a.peak.broad_gauss_u,,,,,
2 d1a.peak.broad_gauss_v,-0.93,,,,
3 d1a.peak.broad_gauss_w,0.79,-0.95,,,
4 d1a.peak.asym_beba_a1,0.28,-0.28,0.27,,
5 d1a.peak.asym_beba_b1,0.32,-0.25,0.19,0.94,


In [36]:
project.display.structure('pbso4')

View(html=<162,946 characters>, height=660)

## Save the project

In [37]:
project.save_as('pd-neut-xray-cwl_pbso4-d1a-lab')
project.report.save_html('pd-neut-xray-cwl_pbso4-d1a-lab.html')

Saving project to 'pd-neut-xray-cwl_pbso4-d1a-lab'
Saving report to 'pd-neut-xray-cwl_pbso4-d1a-lab.html'
